<a href="https://colab.research.google.com/github/RaphaelRAY/projeto-bi-games/blob/main/notebooks/01_importar_dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Ingestão de Dados do Kaggle para o BigQuery
**Dataset: Mercado de Games**

Este notebook é o primeiro passo da nossa arquitetura de BI. Ele automatiza o download, limpeza e envio dos dados para a camada RAW (Bronze) do BigQuery.

## 1. Importação das Bibliotecas

In [ ]:
import pandas as pd
import os
import kagglehub
from google.cloud import bigquery
from google.colab import auth

# Autenticação no Google Colab
auth.authenticate_user()

## 2. Configuração do BigQuery

In [ ]:
project_id = 'directed-mender-489100-m3'
dataset_id = 'games_data'           # Camada RAW
client = bigquery.Client(project=project_id)

## 3. Download dos Dados (Kaggle)

In [ ]:
print("Buscando base de dados no Kaggle...")
data_path = kagglehub.dataset_download("gabrigoncam/video-game-dataset-multidimensional")

all_files = []
for root, dirs, files in os.walk(data_path):
    for file in files:
        if file.endswith('.csv'):
            all_files.append(os.path.join(root, file))

print(f"\n{len(all_files)} arquivos encontrados.")

## 4. Carregamento Inicial (CSV -> DataFrame)

In [ ]:
data_frames = {} # Dicionário para armazenar as tabelas

for file_path in all_files:
    file_name = os.path.basename(file_path)
    table_name = file_name.replace('.csv', '')
    
    print(f"Lendo {file_name}...")
    
    # Tratamentos específicos de leitura
    if file_name == 'games.csv':
        df = pd.read_csv(file_path, engine='python', on_bad_lines='skip')
    elif file_name == 'game_platforms.csv':
        df = pd.read_csv(file_path, dtype={'game_id': str})
    elif file_name == 'game_developers.csv':
        df = pd.read_csv(file_path, engine='python', on_bad_lines='skip')
    else:
        df = pd.read_csv(file_path)
        
    data_frames[table_name] = df

print("\nCarga concluída em memória.")

## 5. Limpeza de Dados e Tratamento de Nulos (NaN)

In [ ]:
for table_name in list(data_frames.keys()):
    print(f"\n--- Limpando Tabela: {table_name} ---")
    df = data_frames[table_name]
    initial_rows = len(df)
    
    # 1. Análise de Nulos (Top 5 colunas com NaN)
    null_info = df.isnull().sum()
    null_info = null_info[null_info > 0].sort_values(ascending=False)
    if not null_info.empty:
        print(f"   >> Colunas com valores nulos encontrados:\n{null_info.head(5)}")
    
    # 2. Remoção de Duplicatas
    df = df.drop_duplicates()
    rows_after_dupes = len(df)
    
    # 3. Tratamento de Nulos Específicos
    # Remover linhas onde ID ou Nome sejam nulos (campos obrigatórios)
    id_col = 'id' if 'id' in df.columns else (df.columns[0] if 'id' in df.columns[0].lower() else None)
    if id_col:
        df = df.dropna(subset=[id_col])
    
    # Preencher notas numéricas com 0 (BI)
    numeric_cols_to_fill = ['metacritic', 'rating', 'playtime', 'ratings_count', 'reviews_count']
    for col in numeric_cols_to_fill:
        if col in df.columns:
            df[col] = df[col].fillna(0)
            
    # Preencher strings vazias com placeholder
    string_cols_to_fill = ['description_short', 'website', 'metacritic_url', 'tba']
    for col in string_cols_to_fill:
        if col in df.columns:
            df[col] = df[col].fillna("Sem Informação")
            
    rows_final = len(df)
    data_frames[table_name] = df
    
    print(f"   >> Duplicatas removidas: {initial_rows - rows_after_dupes}")
    print(f"   >> Linhas com ID nulo removidas: {rows_after_dupes - rows_final}")
    print(f"   >> Total Final: {rows_final} linhas.")

## 6. Exploração (EDA)

In [ ]:
for table_name, df in data_frames.items():
    print(f"\n--- Prévia: {table_name} ---")
    display(df.head(3))
    df.info(verbose=False, memory_usage='deep')

## 7. Ingestão para o BigQuery

In [ ]:
for table_name, df in data_frames.items():
    table_id = f"{project_id}.{dataset_id}.{table_name}"
    print(f"Enviando {table_name} para {table_id}...")
    
    job_config = bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()
    
    print(f"✅ Sucesso! Tabela {table_name} carregada.\n")